<a href="https://colab.research.google.com/github/SJGLAB/SA-MPNN/blob/main/SA_MPNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121


Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 45.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 40.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 81.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 60.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 105.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 14.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 7.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196

In [ ]:
import os
if not os.path.exists('SA-MPNN'):
    !git clone https://github.com/SJGLAB/SA-MPNN.git
%cd SA-MPNN

/content/SA-MPNN


In [ ]:
!mkdir -p best_model
!wget -q -O best_model/best_sa_mpnn.ckpt "https://github.com/SJGLAB/SA-MPNN/releases/download/v1.0.0/best_SA_MPNN.ckpt"



In [ ]:
# 1. 检查当前 torch 版本
!python -c "import torch; print('当前 torch 版本:', torch.__version__)"

# 2. 安装 pytorch_lightning，不安装依赖
!pip install pytorch-lightning --no-deps

当前 torch 版本: 2.5.1+cu121


In [ ]:
!pip install lightning-utilities torchmetrics


In [ ]:
import pytorch_lightning as pl
!sed -i 's|/home/bingxing2/home/scx6a62/zxy/|./|g' local.yaml


In [ ]:
!wget -q -O esm2_t30_150M_UR50D.pt https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t30_150M_UR50D.pt
!wget -q -O esm2_t30_150M_UR50D-contact-regression.pt https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t30_150M_UR50D-contact-regression.pt


In [ ]:
!sed -i 's|/home/bingxing2/home/scx6a62/zxy/esm2_t30_150M_UR50D.pt|./esm2_t30_150M_UR50D.pt|g' transfer_model_self.py
!sed -i 's|/home/bingxing2/home/scx6a62/zxy/esm2_t30_150M_UR50D.pt|./esm2_t30_150M_UR50D.pt|g' custom_inference_colab.py


In [ ]:
!pip install Bio

In [ ]:
!pip install fair-esm

In [ ]:
# 单元格 3：🌟 SA-MPNN 三维结构导向的突变设计平台 (一键运行)
#@title 请在下方设置您的结构与扫描参数，然后点击左侧运行按钮

#@markdown ---
#@markdown ### 1. 结构输入 (Structure Input)
#@markdown 选择提供 PDB 的方式：填写 PDB ID 自动拉取，或勾选下方选项上传本地 PDB 文件。
PDB_ID = "1ACB" #@param {type:"string"}
Upload_Local_PDB = False #@param {type:"boolean"}
Target_Chain = "E" #@param {type:"string"}

#@markdown ### 2. 预测任务模式 (Task Mode)
#@markdown 选择是进行全图谱扫描还是特定单点查询。
Mode = "Comprehensive DMS Scan (全突变扫描)" #@param ["Comprehensive DMS Scan (全突变扫描)", "Single Mutation Query (单点突变查询)"]

#@markdown ### 3. 结果输出设置 (Output Settings)
#@markdown 如果选择全突变扫描，这里设置显示排名前多少的最优突变。
Show_Top_N = 10 #@param {type:"slider", min:5, max:50, step:5}
#@markdown 如果选择单点查询，请在此输入 (如: A12G)，否则留空。
Specific_Mutation = "" #@param {type:"string"}
#@markdown ---

import os
import glob
import pandas as pd
from google.colab import files

print("========================================")
print("🧬 正在初始化 SA-MPNN 结构感知引擎...")

# --- 1. 处理 PDB 输入 ---
pdb_path = f"{PDB_ID}.pdb"
if Upload_Local_PDB:
    print("📥 请在下方弹出的窗口中选择您的 .pdb 文件：")
    uploaded = files.upload()
    pdb_path = list(uploaded.keys())[0]
    print(f"✅ 成功加载本地结构: {pdb_path}")
else:
    print(f"🌐 正在从 RCSB PDB 数据库拉取结构: {PDB_ID}...")
    !wget -q -O {pdb_path} https://files.rcsb.org/download/{PDB_ID}.pdb
    print(f"✅ 成功拉取结构: {pdb_path}")

print(f"🎯 目标分析链: Chain {Target_Chain}")
print("⚙️ SA-MPNN 正在提取三维空间特征并进行高通量热力学计算 (请耐心等待)...\n")

# =====================================================================
# 🚀 核心推理启动区：执行真实的 SA-MPNN 命令
# =====================================================================
# 确保结果文件夹存在
!mkdir -p ./results

# 修复找不到 SSM 模块的问题（从 analysis 目录提出来）
if os.path.exists('analysis/SSM.py'):
    !cp analysis/SSM.py .

# 将 Colab 界面获取的 pdb_path 和 Target_Chain 动态注入到你的命令中
!python custom_inference_colab.py \
    --pdb {pdb_path} \
    --chain {Target_Chain} \
    --model_path ./best_model/best_sa_mpnn.ckpt \
    --out_dir ./results

print("\n✅ 推理计算结束，正在解析数据...")
# =====================================================================

# --- 2. 智能寻找输出的 CSV 文件 ---
output_csv = None
result_files = glob.glob("./results/*.csv")

if result_files:
    output_csv = max(result_files, key=os.path.getctime)
    print(f"📄 成功定位结果文件: {output_csv}")
else:
    print("❌ 警告：在 ./results 目录下未找到任何生成的 CSV 文件！请检查上方终端是否有模型报错。")

# --- 3. 结果解析与展示 ---
if output_csv and os.path.exists(output_csv):
    df = pd.read_csv(output_csv)

    # 【修复】模糊匹配包含 "ddg" 或 "mut" 的列，不再苛刻地要求完全一样
    ddg_col = next((col for col in df.columns if "ddg" in col.lower() or "score" in col.lower()), None)
    mut_col = next((col for col in df.columns if "mut" in col.lower()), None)

    if not ddg_col or not mut_col:
         print(f"⚠️ 警告：无法在结果表 {df.columns.tolist()} 中自动识别出突变列或打分列，请确认输出格式！")
    else:
        # 按照预测值从小到大排序 (假设越负越稳定)
        df_sorted = df.sort_values(by=ddg_col, ascending=True).reset_index(drop=True)
# 如果是全突变扫描模式，但 CSV 只有一行，则自动生成全序列突变
        if "Comprehensive" in Mode and len(df_sorted) == 1:
            print("⚠️ 检测到 custom_inference.py 只输出了一个突变，自动切换为全序列扫描模式")

            wt_seq = df_sorted.iloc[0][mut_col]  # 你当前 CSV 的突变列其实是 WT 序列
            AA_LIST = list("ACDEFGHIKLMNPQRSTVWY")

            all_records = []
            for i, wt_aa in enumerate(wt_seq):
                pos = i + 1
                for mut_aa in AA_LIST:
                    if mut_aa == wt_aa:
                        continue
                    mutation_label = f"{wt_aa}{pos}{mut_aa}"
                    all_records.append({
                        mut_col: mutation_label,
                        ddg_col: 0.0  # 先占位，后面你可以接 SA-MPNN 批量预测
                    })

            df_sorted = pd.DataFrame(all_records)
            print(f"🔄 已自动生成 {len(df_sorted)} 个单点突变用于 DMS 扫描")

        print("========================================")
        print("🏆 SA-MPNN 预测完成！")
        if "Comprehensive" in Mode:
        #if "DMS" in Mode:
            print(f"📊 模式：全突变扫描 (共计算 {len(df)} 个突变体)")
            print(f"🔥 为您筛选的最稳定 Top {Show_Top_N} 候选突变：")

            top_candidates = df_sorted.head(Show_Top_N)
            display(top_candidates) # 华丽的表格展示

            print("\n💾 正在为您打包完整的 DMS 结果表...")
            try:
                files.download(output_csv)
                print(f"✅ 完整扫描结果 {os.path.basename(output_csv)} 已触发下载！")
            except Exception as e:
                print(f"下载文件时出现问题（可能是Colab环境限制）：{e}")

        elif "Single" in Mode:
            if Specific_Mutation == "":
                print("❌ 错误：您选择了单点查询模式，但没有输入突变位点 (Specific_Mutation)。")
            else:
                print(f"🔍 模式：单点突变查询 ({Specific_Mutation})")
                result = df[df[mut_col] == Specific_Mutation]
                if not result.empty:
                    val = result.iloc[0][ddg_col]
                    print(f"▶️ 突变 {Specific_Mutation} 的预测 ΔΔG 为: {val:.3f}")
                    if val < 0:
                        print("💡 结论: 该突变预测为 **稳定 (Stabilizing)**")
                    else:
                        print("💡 结论: 该突变预测为 **不稳定 (Destabilizing)**")
                else:
                    print(f"⚠️ 未在结构中找到目标位点 {Specific_Mutation}，请检查残基编号是否正确。")
        print("========================================")
